In [1]:
%%capture
!pip install transformer_lens

In [ ]:
!hf auth login --token TOKEN

A new version of huggingface_hub (1.9.0) is available! You are using version 1.8.0.
To update, run: pip install -U huggingface_hub

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: write).
The token `Hallucinations` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `Hallucinations`


In [1]:
from transformer_lens import HookedTransformer

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
model = HookedTransformer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loaded pretrained model meta-llama/Llama-3.1-8B-Instruct into HookedTransformer


In [2]:
from datasets import load_dataset
import torch
import re

MAX_SAMPLES = 100
MAX_NEW_TOKENS = 50
TEMPERATURE = 0.1 

truthful_qa = load_dataset(
    "domenicrosati/TruthfulQA", split="train", streaming=True
).take(MAX_SAMPLES)

records = []

for i, data in enumerate(truthful_qa):
    question = data["Question"]
    answer = data["Best Answer"]

    # Tokenize question
    tokens = model.to_tokens(question)

    with torch.no_grad():
        # Generate answer
        generated = model.generate(
            tokens, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE
        )

        # Re-run full sequence with cache to get all hooks
        _, cache = model.run_with_cache(generated)

    # Extract before deleting
    hook_embed = cache["hook_embed"].cpu()
    layer_0_input = cache["blocks.0.hook_resid_pre"].cpu()

    del cache
    torch.cuda.empty_cache()

    model_answer = model.to_string(generated[0, tokens.shape[1] :])

    records.append(
        {
            "metadata": {
                "idx": i,
                "question": question,
                "answer": answer,
                "model_answer": model_answer,
                "question_len": tokens.shape[1],
                "total_len": generated.shape[1],
                "answer_len": generated.shape[1] - tokens.shape[1],
            },
            "hook_embed": hook_embed,
            "hook_pos_embed": layer_0_input,
        }
    )

# ── Save ───────────────────────────────────────────────────────────────────────
out_name = f"{re.sub('/', '_', MODEL_NAME)}_all_layers_with_embeddings.pt"
torch.save(records, out_name)
print(f"\nSaved {len(records)} records → {out_name}")

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

...

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]


Saved 100 records → meta-llama_Llama-3.1-8B-Instruct_all_layers_with_embeddings.pt


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp "qwen3-4b_all_layers_with_attn.pt" /content/drive/MyDrive/
drive.flush_and_unmount()

In [2]:
import torch
records = torch.load(
    "/Users/rachitdas/Desktop/hallucinations_arc/Embedding_analysis/qwen2.5-1.5b_hallucination_labels_embeddings.pt"
)
row = records[0]

embed = row["hook_embed"]
resid = row["hook_pos_embed"]

print("hook_embed shape:", embed.shape)
print("hook_pos_embed shape:", resid.shape)
print("Are they identical?", torch.allclose(embed, resid))
print("Max difference:", (embed - resid).abs().max().item())

hook_embed shape: torch.Size([1, 61, 1536])
hook_pos_embed shape: torch.Size([1, 61, 1536])
Are they identical? True
Max difference: 0.0
